# IMDB Movie Review Sentiment Dataset — GCS Data Preparation

**Project:** Movie Review Sentiment Classifier  
**Dataset:** IMDB 50,000 movie reviews  
**Task:** Binary sentiment classification — `positive` vs `negative`

This notebook prepares the IMDB dataset for downstream machine-learning work and stores the resulting datasets in Google Cloud Storage (GCS).

### Workflow

1. Authenticate with Google Cloud
2. Configure the GCS project and bucket
3. Download the raw IMDB dataset from GCS
4. Validate the dataset
5. Create stratified **80% train / 10% dev / 10% test** splits
6. Create **8-fold stratified cross-validation** folds from the training set only
7. Upload the partitions to GCS
8. Verify the final GCS structure and a sample shard


## 1. Import Libraries and Configure GCP

In [1]:
import os
import pandas as pd

from google.colab import auth
from google.cloud import storage
from sklearn.model_selection import train_test_split, StratifiedKFold

# Authenticate the current Colab session with Google Cloud.
auth.authenticate_user()

# -----------------------------
# GCP configuration
# -----------------------------
PROJECT_ID = "dh-and-infra-for-ai"
BUCKET_NAME = "imdb-sentiment-project"
LOCATION = "EU"

# GCS locations
RAW_GCS_PATH = "raw/IMDB Dataset.csv"
SHARDED_PREFIX = "sharded"

# Local working paths
LOCAL_RAW_PATH = "IMDB Dataset.csv"
LOCAL_SHARDS_FOLDER = "shards"

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{BUCKET_NAME}")
print(f"Raw object: gs://{BUCKET_NAME}/{RAW_GCS_PATH}")


Project: dh-and-infra-for-ai
Bucket: gs://imdb-sentiment-project
Raw object: gs://imdb-sentiment-project/raw/IMDB Dataset.csv


## 2. Connect to the Existing GCS Bucket

This project already uses the bucket `imdb-sentiment-project`. The notebook checks that the bucket is accessible before continuing.


In [2]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

if not storage_client.lookup_bucket(BUCKET_NAME):
    raise ValueError(
        f"Bucket gs://{BUCKET_NAME} was not found or is not accessible."
    )

print(f"Connected successfully to gs://{BUCKET_NAME}")


Connected successfully to gs://imdb-sentiment-project


## 3. Download the Raw IMDB Dataset from GCS


In [3]:
blob = bucket.blob(RAW_GCS_PATH)

if not blob.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at gs://{BUCKET_NAME}/{RAW_GCS_PATH}"
    )

blob.download_to_filename(LOCAL_RAW_PATH)

print(f"Downloaded gs://{BUCKET_NAME}/{RAW_GCS_PATH}")
print(f"Saved locally as: {LOCAL_RAW_PATH}")
print(f"File size: {os.path.getsize(LOCAL_RAW_PATH):,} bytes")


Downloaded gs://imdb-sentiment-project/raw/IMDB Dataset.csv
Saved locally as: IMDB Dataset.csv
File size: 66,497,315 bytes


## 4. Load and Validate the IMDB Dataset

In [4]:
df = pd.read_csv(
    LOCAL_RAW_PATH,
    encoding="utf-8"
)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

expected_columns = {"review", "sentiment"}
missing_columns = expected_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Keep only the expected project columns if extra columns are present.
df = df[["review", "sentiment"]].copy()

# Basic data-quality checks
print("\nMissing values:")
print(df.isna().sum())

print(f"\nDuplicate reviews: {df['review'].duplicated().sum()}")

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())
print("\nSentiment proportions:")
print(df["sentiment"].value_counts(normalize=True).round(4))

display(df.head())


Dataset shape: (50000, 3)
Columns: ['review', 'sentiment', 'split']

Missing values:
review       0
sentiment    0
dtype: int64

Duplicate reviews: 418

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sentiment proportions:
sentiment
positive    0.5
negative    0.5
Name: proportion, dtype: float64


,review,sentiment
0,I caught this little gem totally by accident b...,positive
1,I can't believe that I let myself into this mo...,negative
2,*spoiler alert!* it just gets to me the nerve ...,negative
3,If there's one thing I've learnt from watching...,negative
4,"I remember when this was in theaters, reviews ...",negative


## 5. Create Stratified Train / Dev / Test Splits

The dataset is split into:

- **80% training**
- **10% development/validation**
- **10% test**

Stratification preserves the positive/negative class proportions across the three partitions.

The random seed is fixed at `42` so the split can be reproduced.


In [5]:
RANDOM_STATE = 42

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["sentiment"]
)

# Second split: divide the 20% temporary set equally into
# 10% development and 10% test.
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["sentiment"]
)

print(f"Train: {len(train_df):,} rows ({len(train_df)/len(df):.1%})")
print(f"Dev:   {len(dev_df):,} rows ({len(dev_df)/len(df):.1%})")
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(df):.1%})")


Train: 40,000 rows (80.0%)
Dev:   5,000 rows (10.0%)
Test:  5,000 rows (10.0%)


## 6. Validate the Split Distributions

This check confirms that each partition contains both sentiment classes with approximately the same class proportions.


In [6]:
def show_split_distribution(name, data):
    counts = data["sentiment"].value_counts()
    proportions = data["sentiment"].value_counts(normalize=True)

    print(f"--- {name} ---")
    print(f"Rows: {len(data):,}")
    print("Counts:")
    print(counts.to_dict())
    print("Proportions:")
    print(proportions.round(4).to_dict())
    print()

show_split_distribution("Train", train_df)
show_split_distribution("Dev", dev_df)
show_split_distribution("Test", test_df)


--- Train ---
Rows: 40,000
Counts:
{'positive': 20000, 'negative': 20000}
Proportions:
{'positive': 0.5, 'negative': 0.5}

--- Dev ---
Rows: 5,000
Counts:
{'negative': 2500, 'positive': 2500}
Proportions:
{'negative': 0.5, 'positive': 0.5}

--- Test ---
Rows: 5,000
Counts:
{'negative': 2500, 'positive': 2500}
Proportions:
{'negative': 0.5, 'positive': 0.5}



## 7. Create 8-Fold Stratified Cross-Validation

Cross-validation is generated from the **training set only**.

In [7]:
skf = StratifiedKFold(
    n_splits=8,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_folds = {}

for fold_idx, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["sentiment"])
):
    fold_train = train_df.iloc[train_idx].copy()
    fold_val = train_df.iloc[val_idx].copy()

    cv_folds[fold_idx] = (fold_train, fold_val)

print(f"Number of CV folds: {len(cv_folds)}")

for fold_idx, (fold_train, fold_val) in cv_folds.items():
    print(
        f"Fold {fold_idx}: "
        f"train={len(fold_train):,}, "
        f"validation={len(fold_val):,}"
    )


Number of CV folds: 8
Fold 0: train=35,000, validation=5,000
Fold 1: train=35,000, validation=5,000
Fold 2: train=35,000, validation=5,000
Fold 3: train=35,000, validation=5,000
Fold 4: train=35,000, validation=5,000
Fold 5: train=35,000, validation=5,000
Fold 6: train=35,000, validation=5,000
Fold 7: train=35,000, validation=5,000


## 8. Prepare Local GCS Shards

The final GCS structure will be:

```text
gs://imdb-sentiment-project/
├── raw/
│   └── IMDB Dataset.csv
└── sharded/
    ├── train.csv
    ├── dev.csv
    ├── test.csv
    └── cv_folds/
        ├── fold_0_train.csv
        ├── fold_0_val.csv
        ├── ...
        ├── fold_7_train.csv
        └── fold_7_val.csv
```

The raw dataset remains unchanged.


In [8]:
# Create local output directories.
os.makedirs(f"{LOCAL_SHARDS_FOLDER}/cv_folds", exist_ok=True)

# Save train/dev/test partitions.
train_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/train.csv",
    index=False
)

dev_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/dev.csv",
    index=False
)

test_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/test.csv",
    index=False
)

# Save the eight CV folds.
for fold_idx, (fold_train, fold_val) in cv_folds.items():
    fold_train.to_csv(
        f"{LOCAL_SHARDS_FOLDER}/cv_folds/fold_{fold_idx}_train.csv",
        index=False
    )

    fold_val.to_csv(
        f"{LOCAL_SHARDS_FOLDER}/cv_folds/fold_{fold_idx}_val.csv",
        index=False
    )

print("Local train/dev/test and CV shards created.")


Local train/dev/test and CV shards created.


## 9. Upload Shards to GCS


In [9]:
def upload_folder_to_gcs(local_folder, bucket_name, gcs_prefix):
    """Recursively upload all files under local_folder to GCS."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    uploaded = 0

    for root, _, files in os.walk(local_folder):
        for file_name in sorted(files):
            local_path = os.path.join(root, file_name)

            relative_path = os.path.relpath(
                local_path,
                local_folder
            ).replace(os.sep, "/")

            gcs_blob_path = f"{gcs_prefix}/{relative_path}"

            blob = bucket.blob(gcs_blob_path)
            blob.upload_from_filename(local_path)

            print(f"Uploaded: gs://{bucket_name}/{gcs_blob_path}")
            uploaded += 1

    return uploaded


uploaded_count = upload_folder_to_gcs(
    LOCAL_SHARDS_FOLDER,
    BUCKET_NAME,
    SHARDED_PREFIX
)

print(f"\nUploaded {uploaded_count} shard files successfully.")


Uploaded: gs://imdb-sentiment-project/sharded/dev.csv
Uploaded: gs://imdb-sentiment-project/sharded/test.csv
Uploaded: gs://imdb-sentiment-project/sharded/train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_0_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_0_val.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_1_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_1_val.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_2_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_2_val.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_3_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_3_val.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_4_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_4_val.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_folds/fold_5_train.csv
Uploaded: gs://imdb-sentiment-project/sharded/cv_fold

## 10. Verify the GCS Bucket Structure

This final check lists the objects stored in the bucket and confirms that a CV validation shard can be read directly from GCS.


In [10]:
# List objects in the bucket.
storage_client = storage.Client(project=PROJECT_ID)
blobs = storage_client.list_blobs(BUCKET_NAME)

print(f"--- Objects in gs://{BUCKET_NAME} ---")

object_names = []
for blob in blobs:
    object_names.append(blob.name)
    print(f"{blob.name} ({blob.size:,} bytes)")

# Check the expected key objects.
required_objects = [
    RAW_GCS_PATH,
    f"{SHARDED_PREFIX}/train.csv",
    f"{SHARDED_PREFIX}/dev.csv",
    f"{SHARDED_PREFIX}/test.csv",
    f"{SHARDED_PREFIX}/cv_folds/fold_0_train.csv",
    f"{SHARDED_PREFIX}/cv_folds/fold_0_val.csv",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in object_names
]

if missing_objects:
    raise ValueError(
        f"Missing expected GCS objects: {missing_objects}"
    )

print("\nAll required GCS objects are present.")


--- Objects in gs://imdb-sentiment-project ---
IMDB Dataset.csv (915,945 bytes)
raw/ (0 bytes)
raw/IMDB Dataset.csv (66,497,315 bytes)
sharded/cv_folds/fold_0_train.csv (46,268,600 bytes)
sharded/cv_folds/fold_0_val.csv (6,656,820 bytes)
sharded/cv_folds/fold_1_train.csv (46,405,653 bytes)
sharded/cv_folds/fold_1_val.csv (6,519,767 bytes)
sharded/cv_folds/fold_2_train.csv (46,301,010 bytes)
sharded/cv_folds/fold_2_val.csv (6,624,410 bytes)
sharded/cv_folds/fold_3_train.csv (46,201,735 bytes)
sharded/cv_folds/fold_3_val.csv (6,723,685 bytes)
sharded/cv_folds/fold_4_train.csv (46,355,396 bytes)
sharded/cv_folds/fold_4_val.csv (6,570,024 bytes)
sharded/cv_folds/fold_5_train.csv (46,384,356 bytes)
sharded/cv_folds/fold_5_val.csv (6,541,064 bytes)
sharded/cv_folds/fold_6_train.csv (46,264,764 bytes)
sharded/cv_folds/fold_6_val.csv (6,660,656 bytes)
sharded/cv_folds/fold_7_train.csv (46,296,324 bytes)
sharded/cv_folds/fold_7_val.csv (6,629,096 bytes)
sharded/dev.csv (6,683,109 bytes)
sharded

## 11. Read a CV Shard Directly from GCS

This smoke test verifies that the generated data can be consumed programmatically from GCS.

In [11]:
GCS_CV_PATH = (
    f"gs://{BUCKET_NAME}/"
    f"{SHARDED_PREFIX}/cv_folds/fold_0_val.csv"
)

fold_0_val_df = pd.read_csv(GCS_CV_PATH)

print(f"Read from: {GCS_CV_PATH}")
print(f"Rows loaded: {len(fold_0_val_df):,}")
print(f"Columns: {fold_0_val_df.columns.tolist()}")

display(fold_0_val_df.head(3))


Read from: gs://imdb-sentiment-project/sharded/cv_folds/fold_0_val.csv
Rows loaded: 5,000
Columns: ['review', 'sentiment']


,review,sentiment
0,"I just watched the documentary ""Fog City Maver...",positive
1,"I just watched this movie, by mistake. What a ...",positive
2,Every movie I have PPV'd because Leonard Malti...,negative


## 12. Final Summary

The notebook has now completed the GCS data-preparation workflow:

- Raw IMDB data is stored under `raw/`.
- The dataset is split into **80% train / 10% dev / 10% test** using stratification.
- **8-fold stratified cross-validation** is generated from the training set only.
- Train/dev/test and CV shards are stored under `sharded/`.
- The final GCS structure is verified.
- A CV shard is read directly from GCS.
